In [4]:
from glaciexplo import utils, slope, thickness, velocity, graphics

import pandas as pd
from sklearn.preprocessing import QuantileTransformer

print(f"OGGM version: {utils.get_oggm_version()}")

OGGM version: 1.6.2


# Glacier Data Exploration

## Context

In the context of glacier modelling, there is a large selection of glaciers available. To make it easier to get started, this tool can help you choose glaciers based on various criteria such as location, slope, area, availability of thickness data, and ice surface velocity data.

This notebook provides an example of how glaciers were selected for running simple inversions. The selection is motivated by the fact that the Shallow Ice Approximation (SIA) can be reasonably applied to mountain glaciers with slopes of less than 20% ([Le Meur, E., et al., 2004, Journal of Glaciology](https://doi.org/10.1016/j.crhy.2004.10.001)).

## Initialization

For this analysis, we selected region '11', which corresponds to the European Alps.



In [5]:
utils.setup_oggm_env('p') # create a temporary directory for OGGM to store its data on the parent directory 

region = 11 # European Alps
gdf = utils.fetch_rgi_data(region)

2026-03-27 15:41:18: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-03-27 15:41:18: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-03-27 15:41:18: oggm.cfg: Multiprocessing: using all available processors (N=10)
2026-03-27 15:41:18: oggm.cfg: Multiprocessing switched ON after user settings.


A first broad filtering is applied to avoid having to initialize all the glaciers in the selected region.

Here, we filter glaciers based on the mean slope of their entire surface and their area (in km²).

In [6]:
gdf_filtered = utils.filter_slope_area(gdf, slope_threshold=21, area_threshold=1)

Then we initialize the glaciers directories.

In [7]:
gdirs = utils.process_glacier_directories(gdf_filtered, 
                                          prepro_level=3,
                                          prepro_border=10,
                                          reset=True,
                                          base_url="https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2023.3/elev_bands/W5E5/"
                                          )

Delete all glacier directories? [Y/n] 

2026-03-27 15:41:24: oggm.workflow: init_glacier_directories from prepro level 3 on 208 glaciers.
2026-03-27 15:41:24: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 208 glaciers


We also want the selected glaciers to have available thickness measurements from direct observations (e.g., through GlaThiDa) and reliable surface velocity data from [Millan et al. (2022)](https://doi.org/10.1038/s41561-021-00885-z).

In [8]:
thickness.add_thickness_data(gdirs)
velocity.add_velocity_data(gdirs, error=True)

2026-03-27 15:41:52: oggm.shop.millan22: RuntimeError occurred during task velocity_to_gdir on RGI60-11.00002: There is no velocity data for this glacier


Error occurred while processing RGI60-11.00002: There is no velocity data for this glacier


## Data exploration

Here, we process the data to create a single DataFrame containing all the information needed to analyze and filter the glaciers.

In [9]:
slope_degree_threshold = 21
gdf_merged = graphics.merge_glacier_data(gdirs, gdf_filtered, slope_degree_threshold)

# filter on the glacier that don't have thickness data or velocity data
gdf_merged = gdf_merged.dropna(subset=['thickness_grid_points', 'velocity_avg_error'])

2026-03-27 15:42:22: oggm.shop.millan22: Applying global task compile_millan_statistics on 208 glaciers
2026-03-27 15:42:22: oggm.workflow: Execute entity tasks [millan_statistics] on 208 glaciers


In this new GeoDataFrame, we can extract insightful information such as:
- The percentage of glacier grid cells covered by GlaThiDa thickness data
- The number of grid cells with available GlaThiDa thickness measurements
- The average relative error of velocity by year
- The percentage of the glacier surface with a slope above the selected threshold (in degrees)

With these metrics, we can then filter the most relevant glaciers to include in our analysis.

In [10]:
gdf_merged.columns

Index(['RGIId', 'GLIMSId', 'BgnDate', 'EndDate', 'CenLon', 'CenLat',
       'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope',
       'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging',
       'Linkages', 'Name', 'check_geom', 'geometry',
       'thickness_coverage_percentage', 'thickness_grid_points',
       'percent_slope_above', 'velocity_avg_error', 'relative_avg_error'],
      dtype='str')

In [11]:
# Define the columns of interest to maximize and minimize
cols_maximize = [
    'Area',
    'thickness_grid_points',
    # 'thickness_coverage_percent', 
]
cols_minimize = [
    'percent_slope_above',
    # 'relative_avg_error'           
]
cols = cols_maximize + cols_minimize

A quantile transform is used to map the values of a variable so that their distribution matches a target distribution, such as a uniform distribution. This method is useful for removing the influence of outliers and making features more comparable by aligning their distributions. It works by assigning values according to their sorted rank rather than their actual magnitude.

In [12]:
n_samples = len(gdf_merged)

qt = QuantileTransformer(output_distribution='uniform', n_quantiles=min(1000, n_samples))
normed = pd.DataFrame(qt.fit_transform(gdf_merged[cols]), columns=cols)

normed.index = gdf_merged.index
for col in cols_minimize:
    normed[col] = 1 - normed[col]

**[Optional]** We apply weights to the columns to reflect their relative importance in our analysis.

In [13]:
gdf_merged['score'] = (
    0.1 * normed['Area'] 
    + 0.5 * normed['percent_slope_above'] 
    + 0.4 * normed['thickness_grid_points']
)
gdf_merged['rank'] = gdf_merged['score'].rank(ascending=False).astype(int)

## Results

Thanks to this scoring we are able to select a first shortlist of glaciers to include in our analysis.

In [14]:
# save in a csv the x best scoring glaciers
x = 30
gdf_sorted = gdf_merged.sort_values('score', ascending=False)
gdf_sorted.head(x).to_csv("best_scoring_glaciers.csv", index=False)
gdf_sorted.head(x)[['rank', 'RGIId', 'percent_slope_above', 'Area', 'thickness_grid_points', 'score']]

,rank,RGIId,percent_slope_above,Area,thickness_grid_points,score
116,1,RGI60-11.02072,2.236025,7.717,941.0,0.979070
157,2,RGI60-11.02773,12.621359,14.312,673.0,0.951938
125,3,RGI60-11.02249,4.307938,2.890,396.0,0.893023
46,4,RGI60-11.00746,12.122845,16.624,269.0,0.879845
98,5,RGI60-11.01702,21.106095,13.174,572.0,0.867442
60,6,RGI60-11.00872,17.793696,12.151,394.0,0.866667
158,7,RGI60-11.02774,19.258251,5.421,445.0,0.849612
108,8,RGI60-11.01876,10.833333,5.290,225.0,0.818605
153,9,RGI60-11.02746,19.656992,5.339,349.0,0.806977
168,10,RGI60-11.02819,16.592920,6.706,238.0,0.800775


Let’s visualize the locations of the selected glaciers!

In [15]:
graphics.glaciers_location(gdf_sorted.head(x), outlines=False)